# KAN-Transformer for Building Energy Prediction — Quickstart

End-to-end walkthrough: synthesize data → preprocess → train the proposed model on a small config → evaluate → reproduce Figure 2.

Run cells top-to-bottom. On a CPU laptop the entire notebook finishes in roughly 5–10 minutes with the quick config.

## 1. Imports and seeding

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))  # so the notebook can import the package

import numpy as np
import torch
import torch.nn as nn

from utils.callbacks import set_seed
from utils.config import load_config
from utils.data import synthetic_energy_dataframe, build_dataloaders
from utils.preprocessing import preprocess_energy_dataframe
from utils.metrics import all_metrics
from utils.visualize import plot_predictions_vs_true
from models import KANTransformer

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Synthesize data and preprocess

Generates 90 days of hourly data for four building categories plus five exogenous features. Then linearly interpolates missing values, cleans anomalies, normalizes to [0, 1], and engineers temporal / rolling / degree-day features.

In [ ]:
df = synthetic_energy_dataframe(periods=24*90)
print(df.shape, df.columns.tolist())
df.head()

In [ ]:
splits = preprocess_energy_dataframe(df)
print('train', len(splits.train), 'val', len(splits.val), 'test', len(splits.test))
print('feature_cols (first 10):', splits.feature_cols[:10])

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(splits, window=48, horizon=12, batch_size=32)
x, y = next(iter(train_loader))
print('x:', x.shape, 'y:', y.shape)

## 3. Build the proposed KAN-Transformer

In [ ]:
input_dim = x.shape[-1]
model = KANTransformer(
    input_dim=input_dim,
    num_targets=4,
    horizon=12,
    d_model=64, n_heads=4, num_layers=2,
    kan_hidden=(64, 128, 64), num_splines=4, num_basis=8,
).to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))

## 4. Train

Short training run on the quick config so the notebook stays fast. For full numbers run `python train.py --config configs/default.yaml` from the command line.

In [ ]:
optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
for epoch in range(3):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optim.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optim.step()
        losses.append(loss.item())
    print(f'epoch {epoch}: train_loss={np.mean(losses):.4f}')

## 5. Evaluate on the test split

In [ ]:
model.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.append(model(xb.to(device)).cpu().numpy())
        trues.append(yb.numpy())
y_pred = np.concatenate(preds)
y_true = np.concatenate(trues)
print(all_metrics(y_true, y_pred))

## 6. Plot Figure 2 (predictions vs. truth)

In [ ]:
out = plot_predictions_vs_true(y_true, y_pred, out_path='quickstart_figure2.png')
from IPython.display import Image
Image(filename=str(out))